# Mental Health in the Tech Industry
## CS 210: Data Management for Data Science
**Authors:** Vanij Patel and Sean Lumasag

**Research Question:** What workplace and personal factors predict whether a tech worker will seek mental health treatment, and how does mental health interfere with their work?

**Datasets:** OSMI Mental Health in Tech Surveys — 2014 (1,259 rows) and 2016 (1,433 rows)

In [39]:
#imported libraries
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, roc_auc_score,
                             confusion_matrix, f1_score, accuracy_score)
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

os.makedirs('output_figures', exist_ok=True)

BLUE   = '#4C72B0'
ORANGE = '#DD8452'
GREEN  = '#2ca02c'

print("All libraries loaded.")

All libraries loaded.


---
## Stage 1 — Data Profiling & Ingestion

We load both CSV files, store them in a SQLite database, and generate quality reports covering null rates, data types, and value distributions. This documents the baseline state of the data before any cleaning.


In [40]:
#read the csv files and stored them
df14 = pd.read_csv('2014.csv')
df16 = pd.read_csv('2016.csv')

#load the data into sqlite database
connect = sqlite3.connect('mental_health.db')
df14.to_sql('mental_health_2014', connect, if_exists='replace', index=False)
df16.to_sql('mental_health_2016', connect, if_exists='replace', index=False)
print("Data loaded into SQLite database.")

#print the number of rows and columns in each dataframe
print(f"2014: {df14.shape[0]} rows, {df14.shape[1]} columns")
print(f"2016: {df16.shape[0]} rows, {df16.shape[1]} columns")
print("\nSaved to 'mental_health.db'.")

Data loaded into SQLite database.
2014: 1259 rows, 27 columns
2016: 1433 rows, 63 columns

Saved to 'mental_health.db'.


In [41]:
#check for null values in each dataframe and print columns with >20% nulls
null14 = (df14.isnull().sum() / len(df14) * 100).round(1)
print("2014 Columns with >20% Nulls:")
print(null14[null14 > 20].sort_values(ascending=False))

2014 Columns with >20% Nulls:
comments          87.0
state             40.9
work_interfere    21.0
dtype: float64


In [42]:
#check for null values in each dataframe and print columns with >20% nulls
null16 = (df16.isnull().sum() / len(df16) * 100).round(1)
print("2016 Columns with >20% Nulls:")
print(null16[null16 > 20].sort_values(ascending=False))

2016 Columns with >20% Nulls:
If you have revealed a mental health issue to a client or business contact, do you believe this has impacted you negatively?                                                        90.0
If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?                                                85.8
Is your primary role within your company related to tech/IT?                                                                                                                        81.6
Do you know local or online resources to seek help for a mental health disorder?                                                                                                    80.0
If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to clients or business contacts?                                                        80.0
Do you have medical coverage (private insuran

In [43]:
#print the column names of each dataframe
print("2014 columns:", list(df14.columns))
print(f"\n2016 has {len(df16.columns)} columns written as longer questions.")
print("Example 2016 column:", df16.columns[0])

2014 columns: ['Timestamp', 'Age', 'Gender', 'Country', 'state', 'self_employed', 'family_history', 'treatment', 'work_interfere', 'no_employees', 'remote_work', 'tech_company', 'benefits', 'care_options', 'wellness_program', 'seek_help', 'anonymity', 'leave', 'mental_health_consequence', 'phys_health_consequence', 'coworkers', 'supervisor', 'mental_health_interview', 'phys_health_interview', 'mental_vs_physical', 'obs_consequence', 'comments']

2016 has 63 columns written as longer questions.
Example 2016 column: Are you self-employed?


---
## Stage 2 — Schema Alignment

The two surveys share no matching column names. The 2014 dataset uses short snake_case identifiers while 2016 uses full question text. We manually map each 2016 question to its 2014 equivalent, then add a `survey_year` column to both datasets before merging.


In [44]:
# Manual mapping: 2016 question text → 2014 column name
SCHEMA_MAP = {
    'Are you self-employed?': 'self_employed',
    'How many employees does your company or organization have?': 'no_employees',
    'Is your employer primarily a tech company/organization?': 'tech_company',
    'Does your employer provide mental health benefits as part of healthcare coverage?': 'benefits',
    'Do you know the options for mental health care available under your employer-provided coverage?': 'care_options',
    'Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?': 'wellness_program',
    'Does your employer offer resources to learn more about mental health concerns and options for seeking help?': 'seek_help',
    'Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?': 'anonymity',
    'If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:': 'leave',
    'Do you think that discussing a mental health disorder with your employer would have negative consequences?': 'mental_health_consequence',
    'Do you think that discussing a physical health issue with your employer would have negative consequences?': 'phys_health_consequence',
    'Would you feel comfortable discussing a mental health disorder with your coworkers?': 'coworkers',
    'Would you feel comfortable discussing a mental health disorder with your direct supervisor(s)?': 'supervisor',
    'Have you heard of or observed negative consequences for co-workers who have been open about mental health issues in your workplace?': 'obs_consequence',
    'Would you bring up a mental health issue with a potential employer in an interview?': 'mental_health_interview',
    'Would you be willing to bring up a physical health issue with a potential employer in an interview?': 'phys_health_interview',
    'Do you feel that your employer takes mental health as seriously as physical health?': 'mental_vs_physical',
    'Do you have a family history of mental illness?': 'family_history',
    'Have you ever sought treatment for a mental health issue from a mental health professional?': 'treatment',
    'If you have a mental health issue, do you feel that it interferes with your work when being treated effectively?': 'wi_treated',
    'If you have a mental health issue, do you feel that it interferes with your work when NOT being treated effectively?': 'wi_untreated',
    'What is your age?': 'Age',
    'What is your gender?': 'Gender',
    'What country do you live in?': 'Country',
    'What US state or territory do you live in?': 'state',
    'Do you work remotely?': 'remote_work',
    'How willing would you be to share with friends and family that you have a mental illness?': 'share_friends_family',
}

# Align 2016 dataset to 2014 schema using the mapping
df16_aligned = df16.rename(columns=SCHEMA_MAP).copy()
df16_aligned['survey_year'] = 2016

df14_aligned = df14.copy()
df14_aligned['survey_year'] = 2014

print("Mapped " + str(len(SCHEMA_MAP)) + " columns from 2016 question text to 2014 names.")
print("survey_year column added to both datasets.")

Mapped 27 columns from 2016 question text to 2014 names.
survey_year column added to both datasets.


---
## Stage 3 — Data Cleaning

We address five known data quality issues documented in the proposal:
1. **Age** — filter to 18–75, replace invalid values with the median
2. **Gender** — standardize 70+ freeform entries into Male, Female, Non-binary, Other
3. **Treatment** — encode Yes/No to 1/0 for 2014 (2016 is already binary)
4. **work_interfere** — fill 2014 nulls as Not Applicable; merge 2016's two interference columns using worst-case logic
5. **High-null columns** — drop any 2016 column with >80% missing values

In [45]:
#wrote a function to clean the Age column
def clean_age(df):
    df = df.copy()
    df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
    df.loc[(df['Age'] < 18) | (df['Age'] > 75), 'Age'] = np.nan
    df['Age'] = df['Age'].fillna(df['Age'].median())
    return df

#clean the Age column in both datasets
df14_clean_age = clean_age(df14_aligned)
df16_clean_age = clean_age(df16_aligned)

print("Age column cleaned. Ages are between 18 and 75. Anything else is filled with median.")

Age column cleaned. Ages are between 18 and 75. Anything else is filled with median.


In [46]:
print("2014 Genders")
print(df14['Gender'].value_counts().to_string())

print("\n2016 Genders")
print(df16['What is your gender?'].value_counts().to_string())

2014 Genders
Gender
Male                                              615
male                                              206
Female                                            121
M                                                 116
female                                             62
F                                                  38
m                                                  34
f                                                  15
Make                                                4
Woman                                               3
Male                                                3
Cis Male                                            2
Female                                              2
Man                                                 2
Female (trans)                                      2
Male-ish                                            1
maile                                               1
Trans-female                                        1
Cis Fema

In [47]:
def standardize_gender(gender):
    if pd.isna(gender):
        return 'Other'
    gender = str(gender).lower().strip()

    female_terms = ['female', 'woman', 'girl', 'femail', 'femake', 'cis f', 'cis-woman',
                    'cisgender female', 'female assigned', 'i identify as female',
                    'female or multi', 'female-bodied', 'female/woman', 'trans woman',
                    'trans-female', 'transitioned, m2f', 'other/transfeminine',
                    'transgender woman', 'genderflux demi-girl', 'fem', 'fm', 'f']
    male_terms = ['male', 'man', 'boy', 'maile', 'msle', 'cis m', 'cisdude', 'dude',
                  'sex is male', 'malr', 'mail', 'make', 'mal', 'm', 'mtf']
    nonbinary_terms = ['non-binary', 'nonbinary', 'agender', 'genderqueer', 'genderfluid',
                       'fluid', 'enby', 'trans', 'queer', 'androgyn', 'bigender',
                       'nb masculine', 'genderflux', 'afab', 'male leaning androgynous',
                       'male/genderqueer', 'neuter']
    
    categories = {
        'Female' : female_terms,
        'Male' : male_terms,
        'Non-binary': nonbinary_terms
    }

    for category, terms in categories.items():
        if any(term in gender for term in terms):
            return category
    return 'Other'

df14_clean_age['Gender'] = df14_clean_age['Gender'].apply(standardize_gender)
df16_clean_age['Gender'] = df16_clean_age['Gender'].apply(standardize_gender)

print("2014:", df14_clean_age['Gender'].value_counts())
print("2016:", df16_clean_age['Gender'].value_counts())

2014: Gender
Male          994
Female        252
Non-binary      8
Other           5
Name: count, dtype: int64
2016: Gender
Male          1060
Female         354
Non-binary      14
Other            5
Name: count, dtype: int64


In [48]:
df14_clean_age['treatment'] = df14_clean_age['treatment'].map({'Yes': 1, 'No': 0})

print("2014 Treatment:", df14_clean_age['treatment'].value_counts())
print("2016 Treatment:", df16_clean_age['treatment'].value_counts())

2014 Treatment: treatment
1    637
0    622
Name: count, dtype: int64
2016 Treatment: treatment
1    839
0    594
Name: count, dtype: int64


In [49]:
df14_clean_age['work_interfere'] = df14_clean_age['work_interfere'].fillna('Not Applicable')
interference_rank = {'Often': 4, 'Sometimes': 3, 'Rarely': 2, 'Never': 1, 'Not Applicable': 0}

def get_worst_interference(row):
    treated = row.get('wi_treated', 'Not Applicable')
    untreated = row.get('wi_untreated', 'Not Applicable')
    if interference_rank.get(treated, 0) >= interference_rank.get(untreated, 0):
        return treated
    else:
        return untreated


# 2016 splits work interference into treated/untreated; use the worse value for 2014 alignment.
df16_clean_age['work_interfere'] = df16_clean_age.apply(get_worst_interference, axis=1)
df16_clean_age['work_interfere'] = df16_clean_age['work_interfere'].replace('Not applicable to me', 'Not Applicable')

print("2014 work_interfere:", df14_clean_age['work_interfere'].value_counts().to_dict())
print("2016 work_interfere:", df16_clean_age['work_interfere'].value_counts().to_dict())

2014 work_interfere: {'Sometimes': 465, 'Not Applicable': 264, 'Never': 213, 'Rarely': 173, 'Often': 144}
2016 work_interfere: {'Often': 547, 'Not Applicable': 452, 'Sometimes': 371, 'Rarely': 52, 'Never': 11}


In [50]:
# Drop 2016 columns with >80% nulls
null_pct_16 = df16_clean_age.isnull().sum() / len(df16_clean_age)
high_null = null_pct_16[null_pct_16 > 0.80].index.tolist()
df16_clean_age.drop(columns=[c for c in high_null if c in df16_clean_age.columns], inplace=True)

print(f"Dropped {len(high_null)} high-null columns from 2016.")
print("Columns removed:", high_null)

Dropped 3 high-null columns from 2016.
Columns removed: ['Is your primary role within your company related to tech/IT?', 'If you have revealed a mental health issue to a client or business contact, do you believe this has impacted you negatively?', 'If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?']


---
## Stage 4 — Merging & Validation

We select the shared aligned columns from both datasets and concatenate into a single DataFrame. We then validate by checking row counts, null rates, feature distributions, and class balance

In [51]:
# Normalize country names before merging (2014 uses 'United States of America')
df14_clean_age['Country'] = df14_clean_age['Country'].replace(
    'United States of America', 'United States')
df16_clean_age['Country'] = df16_clean_age['Country'].replace(
    'United States of America', 'United States')

SHARED = ['Age', 'Gender', 'Country', 'self_employed', 'family_history',
          'treatment', 'work_interfere', 'no_employees', 'remote_work',
          'tech_company', 'benefits', 'care_options', 'wellness_program',
          'seek_help', 'anonymity', 'leave', 'mental_health_consequence',
          'phys_health_consequence', 'coworkers', 'supervisor',
          'mental_health_interview', 'phys_health_interview',
          'mental_vs_physical', 'obs_consequence', 'survey_year',
          'share_friends_family']

cols14 = [c for c in SHARED if c in df14_clean_age.columns]
cols16 = [c for c in SHARED if c in df16_clean_age.columns]
df_merged = pd.concat([df14_clean_age[cols14], df16_clean_age[cols16]], ignore_index=True)

print(f"Merged shape: {df_merged.shape}")
print(f"Rows per year: {df_merged['survey_year'].value_counts().to_dict()}")
print(f"Treatment class balance: {df_merged['treatment'].value_counts().to_dict()}")

Merged shape: (2692, 26)
Rows per year: {2016: 1433, 2014: 1259}
Treatment class balance: {1: 1476, 0: 1216}


In [52]:
# Validate: check remaining null counts
print("Remaining null counts (non-zero only):")
print(df_merged.isnull().sum()[df_merged.isnull().sum() > 0])

Remaining null counts (non-zero only):
self_employed                  18
no_employees                  287
tech_company                  287
benefits                      287
care_options                  420
wellness_program              287
seek_help                     287
anonymity                     287
leave                         287
mental_health_consequence     287
phys_health_consequence       287
coworkers                     287
supervisor                    287
mental_vs_physical            287
obs_consequence               287
share_friends_family         1259
dtype: int64


In [53]:
# Save merged dataset to SQLite
df_merged.to_sql('merged_final', connect, if_exists='replace', index=False)
print("Merged dataset saved — SQLite table: merged_final")

Merged dataset saved — SQLite table: merged_final
